# Final Submission Notebook - Setup kỹ + Pipeline rõ từng bước

Notebook này là bản nộp chính thức, kết hợp:

- Setup chi tiết, ổn định cho Colab/local.
- Pipeline tách rõ từng giai đoạn như main_pipeline.

## Pipeline tổng thể

1. Stage 0 - Environment setup
2. Stage 1 - Data ingestion
3. Stage 2 - EDA
4. Stage 3 - Preprocessing
5. Stage 4 - Feature engineering
6. Stage 5 - Model training
7. Stage 6 - Evaluation and comparison
8. Stage 7 - Summary of results
9. Stage 8 - Visualization

Lưu ý:
- Không mount Google Drive/kho cá nhân khi chạy.
- Nếu tải từ Kaggle, cần kaggle.json hoặc biến môi trường Kaggle API.
- Nên chạy tuần tự từ trên xuống dưới.

## Stage 0 - Thiết lập môi trường

Chuẩn bị môi trường chạy trên Colab/local: clone hoặc dùng source sẵn có, xác định project root, sau đó cài dependencies.

### Run Order Map

Thứ tự chạy khuyến nghị:

1. Stage 0: setup môi trường và dependencies
2. Stage 1: chuẩn bị dữ liệu đầu vào
3. Stage 2: thực hiện EDA
4. Stage 3: preprocessing
5. Stage 4: tạo đặc trưng
6. Stage 5: huấn luyện mô hình
7. Stage 6: đánh giá và so sánh
8. Stage 7: tổng kết kết quả
9. Stage 8: visualize outputs

In [ ]:
import os, subprocess, shutil
from pathlib import Path
import zipfile

# Chỉ clone khi chạy trên Colab; trên local thường không cần clone (đã có source)
try:
    import google.colab  # type: ignore
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    files = None

REPO = "https://github.com/TaikiInomata/ML_Assignment_K22_HCMUT_GROUP_6.git"
CLONE_DIR = "project_folder"

cwd = Path.cwd().resolve()

if IN_COLAB:
    if not os.path.exists(CLONE_DIR):
        print(f"🔄 Đang cố gắng clone từ GitHub: {REPO}")
        try:
            subprocess.run(["git", "clone", REPO, CLONE_DIR], check=True, timeout=60)
            print("✅ Git clone thành công!")
        except Exception as clone_error:
            print(f"⚠️  Git clone thất bại: {clone_error}")
            print("💡 Fallback: Vui lòng upload file project_folder.zip từ máy local")
            print("   Hoặc sử dụng Google Drive để mount dữ liệu")
            
            # Thử upload file zip từ Colab
            if files:
                print("\n📤 Chờ upload file zip chứa project...")
                try:
                    uploaded = files.upload()
                    for filename in uploaded.keys():
                        if filename.endswith('.zip'):
                            print(f"✅ Đã nhận file: {filename}")
                            with zipfile.ZipFile(filename, 'r') as zip_ref:
                                zip_ref.extractall('.')
                            os.remove(filename)
                            
                            # Tìm thư mục được extract
                            for item in os.listdir('.'):
                                if os.path.isdir(item) and item != '__pycache__':
                                    CLONE_DIR = item
                                    break
                            print(f"✅ Đã extract project vào: {CLONE_DIR}")
                        else:
                            print(f"⚠️  File {filename} không phải zip, bỏ qua")
                except Exception as upload_error:
                    print(f"❌ Upload thất bại: {upload_error}")
                    print("⚠️  Vui lòng tải file từ: https://github.com/TaikiInomata/ML_Assignment_K22_HCMUT_GROUP_6")
                    raise RuntimeError("Không thể lấy project files. Vui lòng upload hoặc clone thủ công.")
    
    os.chdir(CLONE_DIR)
    project_root = Path.cwd()
    print("✅ Đang chạy trên Colab. Project root:", project_root.as_posix())
else:
    print("Không chạy trên Colab — bỏ qua bước git clone.")
    # Nếu đang trong thư mục 'notebooks', di chuyển lên project root tự động
    if cwd.name.lower() == "notebooks":
        os.chdir(cwd.parent)
        project_root = Path.cwd()
        print("Đã chuyển thư mục làm việc lên project root:", project_root.as_posix())
    else:
        # Cố gắng tự động tìm project root theo dấu hiệu nếu không ở trong notebooks/
        def find_project_root(start: Path) -> Path:
            markers = ["config.py", "requirements.txt", "scripts", "modules"]
            for p in [start, *start.parents]:
                if all((p / m).exists() for m in markers):
                    return p
            return start
        project_root = find_project_root(cwd)
        os.chdir(project_root)
        print("Working directory hiện tại đã được đặt thành:", project_root.as_posix())

# Đặt biến `project_root` toàn cục để các cell sau dùng chung
globals()["project_root"] = project_root

In [ ]:
from pathlib import Path
import sys
import subprocess

# Xác định project root an toàn (nếu notebook đang ở trong thư mục notebooks/ thì lên 1 cấp)
def find_project_root(start: Path) -> Path:
    markers = ["config.py", "requirements.txt", "scripts", "modules"]
    for p in [start, *start.parents]:
        if all((p / m).exists() for m in markers):
            return p
    return start

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else find_project_root(cwd)
req_path = project_root / "requirements.txt"
print("Project root:", project_root.as_posix())

if req_path.exists():
    print("Cài đặt từ:", req_path)
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(req_path)], check=True)
    except subprocess.CalledProcessError as e:
        print("Lỗi khi cài từ requirements.txt:", e)
        # Thử file thay thế đã tạo bởi workflow (không gồm TensorFlow và gói jupyter nặng)
        alt = project_root / 'requirements-no-tf-no-jupyter.txt'
        if alt.exists():
            print("Thử cài các phụ thuộc không gồm TensorFlow/Jupyter từ:", alt.as_posix())
            subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(alt)], check=True)
        else:
            print("Không tìm thấy file thay thế. Gợi ý:")
            print(" - Bật Windows Long Path nếu gặp lỗi đường dẫn dài (pip warning).")
            print(" - Tạo file 'requirements-no-tf-no-jupyter.txt' hoặc cài tay các gói cần thiết.")
else:
    print("Không tìm thấy requirements.txt tại:", req_path.as_posix())
    print("Bạn có thể tạo file requirements.txt tại project root hoặc cài thủ công các thư viện cần thiết.")

## Stage 1.1 - Khai báo nguồn dữ liệu công khai

Xác định nguồn dữ liệu public để đảm bảo truy xuất dữ liệu minh bạch và có thể tái lập.

In [ ]:
PUBLIC_DATA_SOURCE = "https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand"
print("Public data source:", PUBLIC_DATA_SOURCE)

## Stage 1.2 - Tải dữ liệu bằng Kaggle API

Notebook sẽ ưu tiên dùng dữ liệu đã có trong data/. Nếu chưa có, notebook thực hiện luồng tải từ Kaggle bằng kaggle.json hoặc biến môi trường.

In [ ]:
import glob
import os
import shutil
import stat
import subprocess
import zipfile
from pathlib import Path

def find_project_root(start: Path) -> Path:
    markers = ["config.py", "requirements.txt", "scripts", "modules"]
    for p in [start, *start.parents]:
        if all((p / m).exists() for m in markers):
            return p
    return start

project_root = find_project_root(Path.cwd().resolve())
data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)
dataset_csv = data_dir / "hotel_bookings.csv"
print("project_root:", project_root.as_posix())

# 0) Neu da co san file du lieu thi bo qua buoc download Kaggle
if dataset_csv.exists():
    print(f"[Skip] Da tim thay du lieu san co: {dataset_csv.as_posix()}")
else:
    # 1) Cai kaggle CLI
    subprocess.run(["python", "-m", "pip", "install", "-q", "kaggle"], check=True)

    # 2) Chuan bi thu muc ~/.kaggle
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json = kaggle_dir / "kaggle.json"

    # 3) Neu co bien moi truong KAGGLE_USERNAME/KAGGLE_KEY thi tao kaggle.json
    kaggle_user = os.getenv("KAGGLE_USERNAME")
    kaggle_key = os.getenv("KAGGLE_KEY")
    if (not kaggle_json.exists()) and kaggle_user and kaggle_key:
        kaggle_json.write_text(
            '{\n  "username": "' + kaggle_user + '",\n  "key": "' + kaggle_key + '"\n}',
            encoding="utf-8",
        )

    # 4) Neu dang chay tren Colab thi cho upload kaggle.json
    if not kaggle_json.exists():
        try:
            from google.colab import files  # type: ignore

            print("[Info] Colab mode: vui long upload kaggle.json")
            uploaded = files.upload()
            if "kaggle.json" in uploaded:
                with open(kaggle_json, "wb") as f:
                    f.write(uploaded["kaggle.json"])
        except ModuleNotFoundError:
            pass

    # 5) Fallback local: tim kaggle.json o mot so vi tri thong dung
    if not kaggle_json.exists():
        candidates = [
            project_root / "kaggle.json",
            project_root / "data" / "kaggle.json",
            project_root / ".kaggle" / "kaggle.json",
            Path.cwd() / "kaggle.json",
            Path.cwd() / "data" / "kaggle.json",
            Path.cwd() / ".kaggle" / "kaggle.json",
        ]
        for c in candidates:
            if c.exists():
                shutil.copy2(c, kaggle_json)
                break

    # 6) Kiem tra kaggle.json bat buoc phai ton tai
    if not kaggle_json.exists():
        raise FileNotFoundError(
            "Khong tim thay ~/.kaggle/kaggle.json va cung chua co data/hotel_bookings.csv. "
            "Hay lam 1 trong 4 cach: "
            "(1) dat file kaggle.json vao ~/.kaggle/, "
            "(2) dat kaggle.json trong thu muc project, "
            "(3) set bien moi truong KAGGLE_USERNAME va KAGGLE_KEY, "
            "(4) dat san data/hotel_bookings.csv vao thu muc data/."
        )

    # 7) Dat quyen doc token an toan (bo qua neu he dieu hanh khong ho tro)
    try:
        kaggle_json.chmod(stat.S_IRUSR | stat.S_IWUSR)
    except Exception:
        pass

    # 8) Download dataset
    subprocess.run(
        [
            "kaggle",
            "datasets",
            "download",
            "-d",
            "jessemostipak/hotel-booking-demand",
            "-p",
            str(data_dir),
            "--force",
        ],
        check=True,
    )

    # 9) Giai nen
    for z in glob.glob(str(data_dir / "*.zip")):
        with zipfile.ZipFile(z, "r") as f:
            f.extractall(str(data_dir))

print("Files in data/:")
for p in sorted(data_dir.glob("*")):
    print("-", p.as_posix())

## Stage 1.3 - Kiểm tra dữ liệu đầu vào

Xác nhận file data/hotel_bookings.csv tồn tại trước khi chuyển sang bước chạy pipeline mô hình.

In [ ]:
from pathlib import Path

# Dung duong dan theo project_root neu da co tu Cell 8, neu khong thi tu dong suy ra
if "project_root" not in globals():
    def find_project_root(start: Path) -> Path:
        markers = ["config.py", "requirements.txt", "scripts", "modules"]
        for p in [start, *start.parents]:
            if all((p / m).exists() for m in markers):
                return p
        return start

    project_root = find_project_root(Path.cwd().resolve())

p = Path(project_root) / "data" / "hotel_bookings.csv"
assert p.exists(), f"Khong tim thay {p.as_posix()}"
print("OK:", p.as_posix(), "size=", p.stat().st_size)

## Stage 2 - Exploratory Data Analysis (EDA)

Phân tích tổng quan dữ liệu, thống kê thiếu dữ liệu, và sinh artifacts EDA vào reports/eda.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

from IPython.display import display


def find_project_root(start: Path) -> Path:
    markers = ["config.py", "requirements.txt", "scripts", "modules"]
    for p in [start, *start.parents]:
        if all((p / m).exists() for m in markers):
            return p
    return start

project_root = find_project_root(Path.cwd().resolve())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config import CONFIG
from modules.eda import generate_eda_report, plot_missing_values

csv_path = Path(project_root) / "data" / "hotel_bookings.csv"
assert csv_path.exists(), f"Khong tim thay du lieu tai {csv_path.as_posix()}"

df = pd.read_csv(csv_path)
print("Data shape:", df.shape)
display(df.head())

eda_output_dir = Path(project_root) / CONFIG["eda"]["output_dir"]
eda_output_dir.mkdir(parents=True, exist_ok=True)

eda_report = generate_eda_report(df, output_dir=str(eda_output_dir))
missing_plot_path = eda_output_dir / "missing_values_chart.png"
_ = plot_missing_values(df, output_path=str(missing_plot_path))

print("EDA completed")
print("EDA report:", (eda_output_dir / "eda_report.json").as_posix())
print("Missing chart:", missing_plot_path.as_posix())

In [ ]:
missing_focus = eda_report.get("missing_values_analysis", {}).get("target_columns_missing", {})
print("Top missing columns:")
for col, val in sorted(missing_focus.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"- {col}: {val}")

## Stage 3 - Preprocessing

Thực hiện pipeline tiền xử lý: xử lý thiếu dữ liệu, mã hóa, scale; sau đó xuất X, y để dùng cho các stage tiếp theo.

In [ ]:
import numpy as np

from modules.preprocessing import preprocess_pipeline

target_col = CONFIG["data"]["target_column"]
X, y, preprocessing_metadata = preprocess_pipeline(
    df=df,
    target_column=target_col,
    config=CONFIG["preprocessing"],
)

X = np.asarray(X)
y = np.asarray(y)

print("Preprocessing completed")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Imputation:", preprocessing_metadata["imputation"]["method"])
print("Encoding:", preprocessing_metadata["encoding"]["method"])
print("Scaling:", preprocessing_metadata["scaling"]["method"])

## Stage 4 - Feature Engineering

Tạo train/test split, áp dụng PCA (nếu bật trong config), và lưu feature artifacts vào thư mục features/.

In [ ]:
from sklearn.model_selection import train_test_split

from modules.features import apply_pca, save_features

split_cfg = CONFIG["models"]["train_test_split"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=split_cfg["test_size"],
    random_state=split_cfg["random_state"],
    stratify=y,
)

print(f"[Features] Train/Test: {X_train.shape[0]}/{X_test.shape[0]}")

features_cfg = CONFIG["features"]
X_train_pca, pca_info = apply_pca(X_train, features_cfg)

if pca_info.get("enabled", False):
    from sklearn.decomposition import PCA

    pca = PCA(n_components=pca_info["n_components"])
    pca.fit(X_train)
    X_test_pca = pca.transform(X_test)
    print(f"[Features] PCA enabled: {X_train.shape[1]} -> {X_train_pca.shape[1]}")
else:
    X_test_pca = X_test
    print("[Features] PCA disabled")

feature_output = dict(features_cfg["output"])
feature_output["path"] = str(Path(project_root) / "features" / "final_submission_features")
feature_output["format"] = "npy"
save_path = save_features(X_train_pca, y_train, feature_output)

print("[Features] Saved at:", save_path)

## Stage 5 - Huấn luyện mô hình

Huấn luyện các mô hình chính (Logistic Regression, SVM, Random Forest, và MLP nếu bật) để chuẩn bị cho bước đánh giá.

In [ ]:
from modules.models import (
    train_logistic_regression,
    train_svm,
    train_random_forest,
    train_mlp,
)

models_cfg = CONFIG["models"]
models_cfg["mlp"]["enabled"] = True
results = {}

print("[Models] Training...")

if models_cfg.get("logistic_regression", {}).get("enabled", True):
    lr_model, lr_pred = train_logistic_regression(
        X_train_pca,
        y_train,
        X_test_pca,
        params=models_cfg.get("logistic_regression", {}).get("params", {}),
    )
    results["logistic_regression"] = {
        "model": lr_model,
        "predictions": lr_pred,
        "y_proba": lr_model.predict_proba(X_test_pca)[:, 1] if hasattr(lr_model, "predict_proba") else None,
    }
    print("- Logistic Regression done")

if models_cfg.get("svm", {}).get("enabled", True):
    svm_model, svm_pred = train_svm(
        X_train_pca,
        y_train,
        X_test_pca,
        params=models_cfg.get("svm", {}).get("params", {}),
    )
    results["svm"] = {
        "model": svm_model,
        "predictions": svm_pred,
        "y_proba": svm_model.predict_proba(X_test_pca)[:, 1] if hasattr(svm_model, "predict_proba") else None,
    }
    print("- SVM done")

if models_cfg.get("random_forest", {}).get("enabled", True):
    rf_model, rf_pred = train_random_forest(
        X_train_pca,
        y_train,
        X_test_pca,
        params=models_cfg.get("random_forest", {}).get("params", {}),
    )
    results["random_forest"] = {
        "model": rf_model,
        "predictions": rf_pred,
        "y_proba": rf_model.predict_proba(X_test_pca)[:, 1] if hasattr(rf_model, "predict_proba") else None,
    }
    print("- Random Forest done")

if models_cfg.get("mlp", {}).get("enabled", False):
    try:
        mlp_model, mlp_pred = train_mlp(
            X_train_pca,
            y_train,
            X_test_pca,
            params=models_cfg.get("mlp", {}).get("params", {}),
        )
        results["mlp"] = {
            "model": mlp_model,
            "predictions": mlp_pred,
            "y_proba": mlp_model.predict(X_test_pca).flatten() if hasattr(mlp_model, "predict") else None,
        }
        print("- MLP done")
    except Exception as e:
        print("- MLP skipped due to error:", e)

print("Total models trained:", len(results))

## Stage 6 - Đánh giá và so sánh mô hình

Tính metrics, sinh biểu đồ từng mô hình, và tạo biểu đồ so sánh tổng quan.

In [ ]:
from modules.evaluation import evaluate_model, compare_models

eval_cfg = CONFIG["evaluation"]
eval_output_dir = Path(project_root) / eval_cfg["output_dir"]
eval_output_dir.mkdir(parents=True, exist_ok=True)

evaluation_results = {}
for model_name, model_data in results.items():
    eval_result = evaluate_model(
        y_true=y_test,
        y_pred=model_data["predictions"],
        y_proba=model_data.get("y_proba"),
        model_name=model_name,
        output_dir=str(eval_output_dir),
        config=eval_cfg,
    )
    evaluation_results[model_name] = eval_result
    print(f"- {model_name}: Acc={eval_result['accuracy']:.4f}, F1={eval_result['f1']:.4f}")

comparison_path = compare_models(
    evaluation_results,
    output_path=str(eval_output_dir / "model_comparison.png"),
)
print("Comparison chart:", comparison_path)

## Stage 7 - Tổng kết kết quả

Tóm tắt hiệu năng mô hình và xác nhận các artifacts đầu ra chính trước khi nộp.

In [ ]:
summary_df = pd.DataFrame([
    {
        "Model": name,
        "Accuracy": f"{res['accuracy']:.4f}",
        "Precision": f"{res['precision']:.4f}",
        "Recall": f"{res['recall']:.4f}",
        "F1": f"{res['f1']:.4f}",
    }
    for name, res in evaluation_results.items()
])

print("FINAL RESULTS")
display(summary_df)

print("Artifacts:")
print("- EDA:", (Path(project_root) / CONFIG["eda"]["output_dir"]).as_posix())
print("- Features:", (Path(project_root) / "features").as_posix())
print("- Evaluation:", eval_output_dir.as_posix())

## Stage 8 - Visualize

Hiển thị nhanh các biểu đồ trong thư mục reports/evaluation để kiểm tra trực quan kết quả.

In [ ]:
from IPython.display import Image, display

imgs = sorted(eval_output_dir.glob("*.png")) if eval_output_dir.exists() else []
print("Found", len(imgs), "visualization files in", eval_output_dir)

for p in imgs[:12]:
    print("-", p.name)
    display(Image(filename=str(p)))